In [73]:
class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)
        if goals_conceded > 5:
            self.adjust_rating(-2)

        
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)


In [75]:
#Home Team
germany_world_cup_starting_xi_vs_curucao = [
    ["Manuel Neuer", "GK", "Bayern Munich", 83],
    ["Joshua Kimmich", "RB", "Bayern Munich", 88],
    ["Jonathan Tah", "CB", "Bayern Munich", 87],
    ["Nico Schlotterbeck", "CB", "Borussia Dortmund", 87],
    ["Nathaniel Brown", "LB", "Eintracht Frankfurt", 81],
    ["Aleksandar Pavlovic", "CDM", "Bayern Munich", 82],
    ["Felix Nmecha", "CM", "Borussia Dortmund", 82],
    ["Florian Wirtz", "LW", "Liverpool", 85],
    ["Leroy Sane", "RW", "Galatasaray", 79],
    ["Jamal Musiala", "CAM", "Bayern Munich", 85],
    ["Kai Havertz", "ST", "Arsenal", 82],

]
germany_world_cup_bench_vs_curucao= [
    ["Deniz Undav", "ST", "Stuttgart", 81],
    ["Leon Goretzka", "CM", "Bayern Munich", 79],
    ["David Raum", "LB", "RB Leipzig", 80],
    ["Waldemar Anton", "CB", "Borussia Dortmund", 81],
    ["Antonio Rudiger", "CB", "Real Madrid", 84],

]


In [77]:
gk = GoalkeeperProfile("Neuer", "Germany", "goalkeeper", 83)
gk.input_match_stats(
    minutes_played=90,
    saves=1,
    saves_inside_box=1,
    saves_outside_box=0,
    goals_conceded=1,
    xG_faced=0.5,
    goals_prevented=-0.5,
    total_passes=22,
    accurate_passes=19,
    total_long_balls=7,
    accurate_long_balls=4,
    touches=27,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Neuer's match rating: 6.58


In [79]:
player = PlayerProfile("Schlotterbeck", "Germany", "defender", 87)

player.input_match_stats(
    minutes=90,
    goals=1,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=74,
    total_passes=86,
    expected_goals=0.31,
    expected_assists=0.04,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=4,
    interceptions=5,
    ball_recoveries=8,
    dribbled_past=0,
    duels_won=9,
    duels_lost=2,
    ground_duels_won=5,
    ground_duels_total=7,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Schlotterbeck's match rating: 9.50


In [81]:
player = PlayerProfile("Kimmich", "Germany", "defender", 88)

player.input_match_stats(
    minutes=83,
    goals=0,
    assists=2,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=65,
    total_passes=73,
    expected_goals=0.83,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=3,
    total_crosses=4,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=5,
    dribbled_past=2,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kimmich's match rating: 8.60


In [83]:
player = PlayerProfile("Brown", "Germany", "defender", 81)

player.input_match_stats(
    minutes=73,
    goals=1,
    assists=1,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=32,
    total_passes=36,
    expected_goals=0.42,
    expected_assists=0.57,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=2,
    total_crosses=5,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=2,
    duels_lost=5,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Brown's match rating: 8.58


In [85]:
player = PlayerProfile("Tah", "Germany", "defender", 87)

player.input_match_stats(
    minutes=73,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=69,
    total_passes=71,
    expected_goals=0,
    expected_assists=0.23,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=1,
    duels_won=0,
    duels_lost=3,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Tah's match rating: 5.80


In [87]:
player = PlayerProfile("Nmecha", "Germany", "midfielder", 82)

player.input_match_stats(
    minutes=72,
    goals=1,
    assists=0,
    total_shots=4,
    shots_on_target=3,
    accurate_passes=38,
    total_passes=40,
    expected_goals=0.26,
    expected_assists=0.08,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=2,
    duels_won=6,
    duels_lost=3,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=2,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Nmecha's match rating: 8.20


In [89]:
player = PlayerProfile("Musiala", "Germany", "midfielder", 85)

player.input_match_stats(
    minutes=64,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=3,
    accurate_passes=13,
    total_passes=26,
    expected_goals=0.29,
    expected_assists=0.08,
    successful_dribbles=4,
    total_dribbles=5,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=3,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=9,
    duels_lost=5,
    ground_duels_won=9,
    ground_duels_total=14,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=2,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Musiala's match rating: 8.50


In [91]:
player = PlayerProfile("Wirtz", "Germany", "midfielder", 85)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=55,
    total_passes=64,
    expected_goals=0.18,
    expected_assists=0.46,
    successful_dribbles=1,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=8,
    dribbled_past=1,
    duels_won=3,
    duels_lost=4,
    ground_duels_won=3,
    ground_duels_total=7,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Wirtz's match rating: 8.29


In [93]:
player = PlayerProfile("Pavlovic", "Germany", "midfielder", 82)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=67,
    total_passes=72,
    expected_goals=0.4,
    expected_assists=0.33,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=6,
    dribbled_past=1,
    duels_won=5,
    duels_lost=7,
    ground_duels_won=5,
    ground_duels_total=12,
    aerial_duels_won=1,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=6,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Pavlovic's match rating: 7.05


In [95]:
player = PlayerProfile("Havertz", "Germany", "forward", 82)

player.input_match_stats(
    minutes=90,
    goals=2,
    assists=0,
    total_shots=2,
    shots_on_target=2,
    accurate_passes=38,
    total_passes=41,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=6,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=4,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Havertz's match rating: 8.75


In [97]:
player = PlayerProfile("Sane", "Germany", "forward", 79)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=3,
    shots_on_target=0,
    accurate_passes=40,
    total_passes=50,
    expected_goals=0.82,
    expected_assists=0.13,
    successful_dribbles=1,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=2,
    dispossessed=1,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=4,
    duels_lost=6,
    ground_duels_won=4,
    ground_duels_total=9,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sane's match rating: 6.51


In [99]:
player = PlayerProfile("Undav", "Germany", "Fowrard", 81)

player.input_match_stats(
    minutes=26,
    goals=1,
    assists=2,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=8,
    total_passes=12,
    expected_goals=0.5,
    expected_assists=0.34,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=0,
    duels_lost=3,
    ground_duels_won=0,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Undav's match rating: 9.15


In [101]:
player = PlayerProfile("Rudiger", "Germany", "defender", 84)

player.input_match_stats(
    minutes=17,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=14,
    total_passes=17,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=1,
    duels_won=1,
    duels_lost=1,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Rudiger's match rating: 6.31


In [103]:
player = PlayerProfile("Raum", "Germany", "defender", 80)

player.input_match_stats(
    minutes=17,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=5,
    total_passes=7,
    expected_goals=0.03,
    expected_assists=0.07,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Raum's match rating: 6.12


In [105]:
player = PlayerProfile("Anton", "Germany", "defender", 81)

player.input_match_stats(
    minutes=7,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=6,
    total_passes=7,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=2,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Anton's match rating: 6.38


In [107]:
player = PlayerProfile("Goretzka", "Germany", "midfielder", 79)

player.input_match_stats(
    minutes=17,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=11,
    total_passes=13,
    expected_goals=0,
    expected_assists=0.04,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=2,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=1,
    duels_won=1,
    duels_lost=3,
    ground_duels_won=0,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Goretzka's match rating: 5.89


In [111]:
#AwayTeam
curucao_world_cup_starting_xi_vs_germany = [
    ["Eloy Room", "GK", "Miami FC", 69],
    ["Riechedly Bazoer", "CB", "Konyaspor", 74],
    ["Armando Obispo", "CB", "PSV Eindhoven", 73],
    ["Deveron Fonville", "RB", "NEC Nijmegen", 69],
    ["Sherel Floranus", "LB", "PEC Zwolle", 68],
    ["Juninho Bacuna", "CM", "FC Volendam", 73],
    ["Leandro Bacuna", "CM", "Igdır", 73],
    ["Livano Comenencia", "CM", "FC Zurich", 73],
    ["Jurgen Locadia", "CAM", "Miami FC", 73],
    ["Tahith Chong", "ST", "Sheffield United", 73],
    ["Sontje Hansen", "st", "Middlesbrough", 72],

]
curucao_world_cup_bench_vs_germany = [
    ["Gervane Kastaneer", "ST", "Terengganu FC", 70],
    ["Jearl Margaritha", "ST", "SK Beveren", 66],
    ["Jeremy Antonisse", "LW", "AE Kifisia", 69],

]   

In [113]:
gk = GoalkeeperProfile("Room", "Curucao", "goalkeeper", 69)
gk.input_match_stats(
    minutes_played=90,
    saves=4,
    saves_inside_box=3,
    saves_outside_box=1,
    goals_conceded=7,
    xG_faced=5.24,
    goals_prevented=-1.76,
    total_passes=14,
    accurate_passes=28,
    total_long_balls=18,
    accurate_long_balls=4,
    touches=44,
    errors=2,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Room's match rating: 6.40


In [117]:
player = PlayerProfile("Floranus", "Curucao", "defender", 68)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=29,
    total_passes=34,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=4,
    dribbled_past=1,
    duels_won=3,
    duels_lost=2,
    ground_duels_won=3,
    ground_duels_total=5,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=7 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Floranus's match rating: 4.56


In [119]:
player = PlayerProfile("Bazoer", "Curucao", "defender", 74)

player.input_match_stats(
    minutes=90,
    conceded_penalty=1,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=31,
    total_passes=35,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=4,
    total_long_balls=8,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=5,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=7 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Bazoer's match rating: 1.79


In [121]:
player = PlayerProfile("Obispo", "Curucao", "defender", 73)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=37,
    total_passes=41,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=3,
    interceptions=3,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=3,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=7 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Obispo's match rating: 5.45


In [123]:
player = PlayerProfile("Fonville", "Curucao", "defender", 69)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=20,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=2,
    interceptions=2,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=4,
    duels_lost=4,
    ground_duels_won=4,
    ground_duels_total=8,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=7 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Fonville's match rating: 5.41


In [125]:
player = PlayerProfile("Commenencia", "Curucao", "midfielder", 73)

player.input_match_stats(
    minutes=90,
    goals=1,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=21,
    total_passes=26,
    expected_goals=0.09,
    expected_assists=0.05,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=2,
    interceptions=4,
    ball_recoveries=5,
    dribbled_past=0,
    duels_won=4,
    duels_lost=2,
    ground_duels_won=3,
    ground_duels_total=5,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=7 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Commenencia's match rating: 7.74


In [127]:
player = PlayerProfile("Leandro Bacuna", "Curucao", "midfielder", 73)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=0,
    accurate_passes=49,
    total_passes=56,
    expected_goals=0.19,
    expected_assists=0.01,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=3,
    dispossessed=1,
    tackles_won=4,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=2,
    duels_won=9,
    duels_lost=5,
    ground_duels_won=7,
    ground_duels_total=11,
    aerial_duels_won=2,
    aerial_duels_total=3,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=7 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Leandro Bacuna's match rating: 5.41


In [129]:
player = PlayerProfile("Juninho Bacuna", "Curucao", "midfielder", 73)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=28,
    total_passes=33,
    expected_goals=0.02,
    expected_assists=0.02,
    successful_dribbles=3,
    total_dribbles=7,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=2,
    total_long_balls=2,
    dispossessed=4,
    tackles_won=3,
    tackles_loss=0,
    clearances=1,
    interceptions=5,
    ball_recoveries=9,
    dribbled_past=1,
    duels_won=8,
    duels_lost=11,
    ground_duels_won=8,
    ground_duels_total=19,
    aerial_duels_won=0,
    aerial_duels_total=2,
    fouled=2,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Juninho Bacuna's match rating: 7.49


In [131]:
player = PlayerProfile("Chong", "Curucao", "midfielder", 73)

player.input_match_stats(
    minutes=83,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=13,
    total_passes=19,
    expected_goals=0.05,
    expected_assists=0.01,
    successful_dribbles=5,
    total_dribbles=6,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=1,
    duels_won=13,
    duels_lost=4,
    ground_duels_won=13,
    ground_duels_total=17,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=8,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Chong's match rating: 6.95


In [133]:
player = PlayerProfile("Hansen", "Curucao", "forward", 72)

player.input_match_stats(
    minutes=45,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=6,
    total_passes=7,
    expected_goals=0.03,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=2,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=2,
    duels_won=1,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=2,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Hansen's match rating: 6.29


In [135]:
player = PlayerProfile("Locadia", "Curucao", "forard", 73)

player.input_match_stats(
    minutes=65,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=13,
    total_passes=14,
    expected_goals=0,
    expected_assists=0.03,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=3,
    duels_lost=4,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=1,
    aerial_duels_total=3,
    fouled=3,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Locadia's match rating: 6.75


In [63]:
player = PlayerProfile("Antoinisse", "Curucao", "forward", 69)

player.input_match_stats(
    minutes=45,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=21,
    expected_goals=0,
    expected_assists=0.13,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=1,
    total_crosses=3,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Antoinisse's match rating: 6.16


In [137]:
player = PlayerProfile("Margaritha", "Curucao", "forward", 66)

player.input_match_stats(
    minutes=25,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=4,
    total_passes=5,
    expected_goals=0.04,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Margaritha's match rating: 5.19


In [139]:
player = PlayerProfile("Kastaneer", "Curucao", "forward", 70)

player.input_match_stats(
    minutes=7,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=3,
    total_passes=4,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=1,
    duels_won=0,
    duels_lost=3,
    ground_duels_won=0,
    ground_duels_total=1,
    aerial_duels_won=0,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kastaneer's match rating: 5.89
